# RDT-1B Feature-level BlurIG
**What this does:** Runs ManiSkill simulation to get a real robot camera frame, then applies Blur Integrated Gradients in SigLIP patch-embedding space to show which parts of the image most influenced RDT's predicted gripper action.

**Before running:** Set `Runtime > Change runtime type > A100 GPU`

**Runtime:** ~5-10 min on A100 (first run downloads ~2GB of models)

In [ ]:
import os
if os.path.exists('/content/.deps_installed'):
    print('Already installed — skip to Cell 2.')
else:
    os.system('git clone https://github.com/thu-ml/RoboticsDiffusionTransformer')
    os.system('git clone https://github.com/rdgbrandon/rdt-igtesting')
    os.system('apt-get install -qq libvulkan1 libegl1-mesa-dev libgles2-mesa-dev libosmesa6-dev')
    os.system('pip install -q einops timm pyyaml mani-skill accelerate')
    os.system('pip install -q "numpy<2"')
    os.system('pip install -q "protobuf>=5.28.0"')
    open('/content/.deps_installed', 'w').close()
    print('Install done. Restarting runtime...')
    os.kill(os.getpid(), 9)

In [ ]:
from huggingface_hub import hf_hub_download
import shutil, os

path = hf_hub_download(
    repo_id='robotics-diffusion-transformer/maniskill-model',
    filename='lang_embeds/text_embed_PickCube-v1.pt',
)
dest = '/content/rdt-igtesting/lang_embed.pt'
os.makedirs(os.path.dirname(dest), exist_ok=True)
shutil.copy(path, dest)
print('Language embedding ready.')

In [ ]:
import os, traceback
os.environ['RDT_REPO']       = '/content/RoboticsDiffusionTransformer'
os.environ['LANG_EMBED']     = 'lang_embed.pt'
os.environ['MANISKILL_TASK'] = 'PickCube-v1'
os.chdir('/content/rdt-igtesting')
!git pull -q

try:
    %run rdt_blurig.py
    print("\nDone — see rdt_blurig_output.png")
except Exception as _e:
    _tb, _msg = traceback.format_exc(), str(_e)
    print(f"\nFAILED: {type(_e).__name__}: {_msg}")
    fixes = []
    if "CUDA out of memory" in _msg:
        fixes.append("OOM — reduce N_BLURIG_STEPS or N_DDPM_STEPS at top of rdt_blurig.py")
    if "ModuleNotFoundError" in _msg:
        mod = _msg.split("'")[1] if "'" in _msg else "?"
        fixes.append(f"Missing '{mod}' — add to Cell 1 pip installs and re-run from Cell 1")
    if "Vulkan" in _tb or "sapien" in _tb.lower():
        fixes.append("Renderer error — make sure GPU runtime is selected")
    print(("\nFix: " + fixes[0]) if fixes else "\n" + _tb)

In [ ]:
from IPython.display import Image
Image('rdt_blurig_output.png')

In [ ]:
import torch, matplotlib.pyplot as plt, numpy as np
from huggingface_hub import hf_hub_download

null_tasks = {
    "push cube":    "PushCube-v1",
    "stack cubes":  "StackCube-v1",
    "insert peg":   "PegInsertionSide-v1",
}

null_results = {}
for task_desc, task_id in null_tasks.items():
    path = hf_hub_download(
        repo_id='robotics-diffusion-transformer/maniskill-model',
        filename=f'lang_embeds/text_embed_{task_id}.pt',
    )
    ld = torch.load(path, map_location="cpu", weights_only=False)
    lt = ld if not isinstance(ld, dict) else ld["embeddings"]
    if lt.dim() == 2:
        lt = lt.unsqueeze(0)
    lt = lt.to(DEVICE, dtype=DTYPE)
    lm = torch.ones(lt.shape[:2], dtype=torch.bool, device=DEVICE)
    with torch.no_grad():
        lc = rdt.lang_adaptor(lt)

    _lc, _lm = lang_cond, lang_attn_mask
    lang_cond, lang_attn_mask = lc, lm
    print(f"BlurIG: same frame + '{task_desc}' ...")
    null_results[task_desc] = feature_blur_ig(single_emb)
    lang_cond, lang_attn_mask = _lc, _lm

all_cases = {TASK_TEXT: attr} | null_results
img_np    = np.array(frame_pil) / 255.0
n         = len(all_cases)

fig, axes = plt.subplots(2, n, figsize=(4.5 * n, 8))
for col, (name, at) in enumerate(all_cases.items()):
    amap = to_map(at, 384, 384)
    axes[0, col].imshow(img_np)
    axes[0, col].set_title(name, fontsize=10)
    axes[1, col].imshow(img_np)
    axes[1, col].imshow(amap, cmap="inferno", alpha=0.6, vmin=0, vmax=1)
    axes[1, col].set_title("attribution", fontsize=9)
    if col == 0:
        axes[0, col].set_xlabel("← real task", fontsize=8)

for ax in axes.flat:
    ax.axis("off")

fig.suptitle(
    f"RDT-1B BlurIG — same frame, different task language\n"
    f"Image: {TASK} scene | Does attribution shift when the task doesn't match?",
    fontsize=10,
)
plt.tight_layout()
plt.savefig("rdt_blurig_null_comparison.png", dpi=150, bbox_inches="tight")
print("Saved: rdt_blurig_null_comparison.png")

from IPython.display import Image as IPyImage
IPyImage("rdt_blurig_null_comparison.png")

---
## Optional: try other tasks
Available tasks and their pre-encoded lang embeds:
- `PickCube-v1` — pick up the cube
- `StackCube-v1` — stack one cube on another
- `PushCube-v1` — push the cube to a target
- `PegInsertionSide-v1` — insert a peg into a hole
- `PlugCharger-v1` — plug a charger into a socket

In [ ]:
from huggingface_hub import hf_hub_download
import shutil, os

TASK = 'StackCube-v1'

path = hf_hub_download(
    repo_id='robotics-diffusion-transformer/maniskill-model',
    filename=f'lang_embeds/text_embed_{TASK}.pt',
)
dest = '/content/rdt-igtesting/lang_embed.pt'
os.makedirs(os.path.dirname(dest), exist_ok=True)
shutil.copy(path, dest)

os.environ['MANISKILL_TASK'] = TASK
os.chdir('/content/rdt-igtesting')
%run rdt_blurig.py

from IPython.display import Image
Image('rdt_blurig_output.png')

---
## Tuning BlurIG parameters
Edit these at the top of `rdt_blurig.py` before running Cell 3:

| Parameter | Default | Effect |
|---|---|---|
| `N_BLURIG_STEPS` | 20 | More steps = smoother heatmap, slower |
| `N_DDPM_STEPS` | 5 | More steps = more faithful to full RDT, slower |
| `SIGMA_MAX` | 2.0 | Higher = blurrier baseline, stronger contrast in heatmap |
| `SCORE_HORIZON` | 8 | How many future action steps to include in the score |